In [90]:
%cd /content
!git clone https://github.com/23064088/DataMiningGroup18.git
%cd DataMiningGroup18
!git checkout random_forest
!git status


/content
fatal: destination path 'DataMiningGroup18' already exists and is not an empty directory.
/content/DataMiningGroup18
Already on 'random_forest'
Your branch is ahead of 'origin/random_forest' by 1 commit.
  (use "git push" to publish your local commits)
On branch random_forest
Your branch is ahead of 'origin/random_forest' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [95]:
import pandas as pd
import numpy as np

df = pd.read_csv("financial_data.csv")
display(df.head())

# Drop ID column
df = df.drop(columns=["Customer_ID"])

TARGET = "Risk_Level"

print(df[TARGET].value_counts())


,Customer_ID,Age,Income,Credit_Score,Investment_Returns,Risk_Level,Customer_Feedback
0,1001,25.0,0.735552,0.127660,11.57,Medium,The app is found.
1,1002,29.0,0.530047,0.157447,7.75,Medium,The service was discontinued.
2,1003,25.0,0.293007,0.421277,11.80,High,Fees are high.
3,1004,41.0,1.000000,0.953191,9.46,Low,Fees are high.
4,1005,25.0,0.934614,0.221277,8.08,High,The app is found.


Risk_Level
Low       419
High      300
Medium    281
Name: count, dtype: int64


In [97]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric:", num_cols)
print("Categorical:", cat_cols)


Numeric: ['Age', 'Income', 'Credit_Score', 'Investment_Returns']
Categorical: ['Customer_Feedback']


In [105]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Income',
                                                   'Credit_Score',
                                                   'Investment_Returns']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Customer_Feedback'])])),
                ('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [126]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.47

Confusion Matrix:
 [[21 29 10]
 [12 53 19]
 [16 20 20]]

Classification Report:
               precision    recall  f1-score   support

        High       0.43      0.35      0.39        60
         Low       0.52      0.63      0.57        84
      Medium       0.41      0.36      0.38        56

    accuracy                           0.47       200
   macro avg       0.45      0.45      0.45       200
weighted avg       0.46      0.47      0.46       200



In [127]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_features = ohe.get_feature_names_out(cat_cols)

feature_names = num_cols + list(cat_features)
importances = model.named_steps["rf"].feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(fi.head(15))


,feature,importance
1,Income,0.278971
2,Credit_Score,0.246348
3,Investment_Returns,0.245960
0,Age,0.122301
7,Customer_Feedback_Happy with the app.,0.020989
5,Customer_Feedback_Fees are high.,0.020169
9,Customer_Feedback_The service was discontinued.,0.017682
4,Customer_Feedback_Average.,0.016291
6,Customer_Feedback_Great service!,0.015897
8,Customer_Feedback_The app is found.,0.015392


In [139]:
from sklearn.metrics import f1_score

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

with open("random_forest_results.txt", "w") as f:
    f.write("Random Forest Results\n")
    f.write(f"Target: {TARGET}\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"F1-score (macro): {f1:.4f}\n\n")
    f.write("Top 10 Features:\n")
    for _, row in fi.head(10).iterrows():
        f.write(f"- {row['feature']}: {row['importance']:.4f}\n")

print("Saved random_forest_results.txt")


Saved random_forest_results.txt
